# Supervised Fine-Tuning with SFTTrainer

This notebook demonstrates how to fine-tune the `HuggingFaceTB/SmolLM2-135M` model using the `SFTTrainer` from the `trl` library. The notebook cells run and will finetune the model. You can select your difficulty by trying out different datasets.

<div style='background-color: lightblue; padding: 10px; border-radius: 5px; margin-bottom: 20px; color:black'>
    <h2 style='margin: 0;color:blue'>Exercise: Fine-Tuning SmolLM2 with SFTTrainer</h2>
    <p>Take a dataset from the Hugging Face hub and finetune a model on it. </p>
    <p><b>Difficulty Levels</b></p>
    <p>🐢 Use the `HuggingFaceTB/smoltalk` dataset</p>
    <p>🐕 Try out the `bigcode/the-stack-smol` dataset and finetune a code generation model on a specific subset `data/python`.</p>
    <p>🦁 Select a dataset that relates to a real world use case your interested in</p>
</div>

In [1]:
# Install the requirements in Google Colab
!pip install transformers datasets trl huggingface_hub

# Authenticate to Hugging Face

from huggingface_hub import login
from google.colab import userdata
login(userdata.get('HF_TOKEN'))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.4/293.4 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 18.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [2]:
# set device.
# if using 'cpu' the training can take hours to complete.
from torch import cuda, backends

device = (
    "cuda"
    if cuda.is_available()
    else "mps" if backends.mps.is_available() else "cpu"
)
print(f"Using device: {device}")

Using device: cuda


[AutoModelForCasualLM](https://huggingface.co/docs/transformers/model_doc/auto#transformers.AutoModelForCausalLM) is a [transformer](https://huggingface.co/docs/transformers/philosophy#philosophy) used in Natual Language Processing.

In [3]:
# Import necessary libraries
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load the model and tokenizer
model_name = "HuggingFaceTB/SmolLM2-135M"
model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=model_name
).to(device)
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_name)


config.json:   0%|          | 0.00/704 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.66k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

Como o tokenizer não tem chat_template não podemos usar `tokenizer.apply_chat_template`

In [8]:
print(tokenizer.chat_template)

None


In [5]:
#messages = [{"role": "user", "content": "Write a haiku about programming"}]
#input_text=tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

ValueError: Cannot use chat template functions because tokenizer.chat_template is not set and no template argument was passed! For information about writing templates and setting the tokenizer.chat_template attribute, please see the documentation at https://huggingface.co/docs/transformers/main/en/chat_templating

Poderíamos usar um `prompt`. Mas não é isso o que queremos.
TODO: explicar melhor a diferença entre prompt e chat.

In [9]:
prompt="Write a haiku about programming"
inputs = tokenizer.encode(prompt, return_tensors="pt").to(device)
outputs = model.generate(inputs, max_new_tokens=100)
print(tokenizer.decode(outputs[0]))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Write a haiku about programming.

A haiku is a short poem that consists of just three lines. The first line is called the “zenzen” and is usually about nature. The second line is called the “ten-zenzen” and is about the beauty of nature. The third line is called the “5-zenzen” and is about the beauty of nature.

A haiku is a very simple poem. It is a way of expressing a feeling or a thought. It is a way


Use `setup_chat_format` to [add special tokens for chat format](https://huggingface.co/docs/trl/en/sft_trainer#add-special-tokens-for-chat-format).

In [10]:
from trl import setup_chat_format
# Set up the chat format
model, tokenizer = setup_chat_format(model=model, tokenizer=tokenizer)


In [11]:
print(tokenizer.chat_template)

{% for message in messages %}{{'<|im_start|>' + message['role'] + '
' + message['content'] + '<|im_end|>' + '
'}}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant
' }}{% endif %}


# Generate with the base model

Here we will try out the base model which does not have a chat template.

In [12]:
# Let's test the base model before training
prompt = "Write a haiku about programming"

# Format with template
messages = [{"role": "user", "content": prompt}]
formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False)

# Generate response
inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)
outputs = model.generate(**inputs, max_new_tokens=100)
print("Before training:")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Before training:
user
Write a haiku about programming
Write a haiku about programming
Write a haiku about programming
Write a haiku about programming
Write a haiku about programming
Write a haiku about programming
Write a haiku about programming
Write a haiku about programming
Write a haiku about programming
Write a haiku about programming
Write a haiku about programming
Write a haiku about programming
Write a haiku about programming
Write a haiku about programming
Write a haiku about programming
Write a


## Dataset Preparation

We will load a [sample dataset](https://huggingface.co/datasets/HuggingFaceTB/smoltalk) and format it for training. The dataset should be structured with input-output pairs, where each input is a prompt and the output is the expected response from the model.

**TRL will format input messages based on the model's chat templates.** They need to be represented as a list of dictionaries with the keys: `role` and `content`,.

In [13]:
from datasets import load_dataset

# TODO: define your dataset and config using the path and name parameters
ds = load_dataset(path="HuggingFaceTB/smoltalk", name="everyday-conversations")

README.md:   0%|          | 0.00/9.25k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/946k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/52.6k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2260 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/119 [00:00<?, ? examples/s]

In [15]:
ds['train'][0]

{'full_topic': 'Travel/Vacation destinations/Beach resorts',
 'messages': [{'content': 'Hi there', 'role': 'user'},
  {'content': 'Hello! How can I help you today?', 'role': 'assistant'},
  {'content': "I'm looking for a beach resort for my next vacation. Can you recommend some popular ones?",
   'role': 'user'},
  {'content': "Some popular beach resorts include Maui in Hawaii, the Maldives, and the Bahamas. They're known for their beautiful beaches and crystal-clear waters.",
   'role': 'assistant'},
  {'content': 'That sounds great. Are there any resorts in the Caribbean that are good for families?',
   'role': 'user'},
  {'content': 'Yes, the Turks and Caicos Islands and Barbados are excellent choices for family-friendly resorts in the Caribbean. They offer a range of activities and amenities suitable for all ages.',
   'role': 'assistant'},
  {'content': "Okay, I'll look into those. Thanks for the recommendations!",
   'role': 'user'},
  {'content': "You're welcome. I hope you find

In [ ]:
# TODO: 🦁 If your dataset is not in a format that TRL can convert to the chat template, you will need to process it. Refer to the [module](../chat_templates.md)

## Configuring the SFTTrainer

The `SFTTrainer` is configured with various parameters that control the training process. These include the number of training steps, batch size, learning rate, and evaluation strategy. Adjust these parameters based on your specific requirements and computational resources.

In [17]:
from trl import SFTConfig, SFTTrainer

# Set our name for the finetune to be saved &/ uploaded to
finetune_name = "SmolLM2-FT-MyDataset"

# Configure the SFTTrainer
# report_to="none" to disable wandb
sft_config = SFTConfig(
    output_dir="./sft_output",
    max_steps=1000,  # Adjust based on dataset size and desired training duration
    per_device_train_batch_size=4,  # Set according to your GPU memory capacity
    learning_rate=5e-5,  # Common starting point for fine-tuning
    logging_steps=10,  # Frequency of logging training metrics
    save_steps=100,  # Frequency of saving model checkpoints
    evaluation_strategy="steps",  # Evaluate the model at regular intervals
    eval_steps=50,  # Frequency of evaluation
    use_mps_device=(
        True if device == "mps" else False
    ),  # Use MPS for mixed precision training
    hub_model_id=finetune_name,  # Set a unique name for your model
    report_to="none"
)

# Initialize the SFTTrainer
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=ds["train"],
    tokenizer=tokenizer,
    eval_dataset=ds["test"]
)

# TODO: 🦁 🐕 align the SFTTrainer params with your chosen dataset. For example, if you are using the `bigcode/the-stack-smol` dataset, you will need to choose the `content` column`

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-17-cf2d1c7e1586>:25: FutureWarning: `tokenizer` is deprecated and removed starting from version 0.16.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  trainer = SFTTrainer(


Map:   0%|          | 0/2260 [00:00<?, ? examples/s]

Map:   0%|          | 0/119 [00:00<?, ? examples/s]

## Training the Model

With the trainer configured, we can now proceed to train the model. The training process will involve iterating over the dataset, computing the loss, and updating the model's parameters to minimize this loss.

In [18]:
# Train the model
trainer.train()

# Save the model
trainer.save_model(f"./{finetune_name}")

Step,Training Loss,Validation Loss
50,1.065700,1.158981
100,1.111600,1.124065
150,1.062400,1.095485
200,1.048200,1.079699
250,1.041200,1.070457
300,1.029200,1.061473
350,1.003400,1.054751
400,1.006500,1.050794
450,1.021100,1.042637
500,1.076200,1.033726


In [19]:
finetune_tags = ["smol-course", "module_1"]
trainer.push_to_hub(tags=finetune_tags)

HfHubHTTPError: (Request ID: Root=1-677d8863-76134dc904032f4835ee7930;fbf17ab4-b270-43fa-a77a-7814a4b00359)

403 Forbidden: You don't have the rights to create a model under the namespace "marcos-banik".
Cannot access content at: https://huggingface.co/api/repos/create.
Make sure your token has the correct permissions.

In [48]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


<div style='background-color: lightblue; padding: 10px; border-radius: 5px; margin-bottom: 20px; color:black'>
    <h2 style='margin: 0;color:blue'>Bonus Exercise: Generate with fine-tuned model</h2>
    <p>🐕 Use the fine-tuned to model generate a response, just like with the base example..</p>
</div>

In [21]:
# Test the fine-tuned model on the same prompt

# Let's test the base model before training
prompt = "Write a haiku about programming"

# Format with template
messages = [{"role": "user", "content": prompt}]
formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False)

# Generate response
inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)

# TODO: use the fine-tuned to model generate a response, just like with the base example.

In [22]:
outputs = model.generate(**inputs, max_new_tokens=1000)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

user
Write a haiku about programming
assistant
Hello! How can I help you today? I'm going to write a haiku about programming. What's programming? It's a type of art where you create a sequence of steps to solve a problem. Think of it like a recipe. You follow the steps, and then you see the result. Can you think of any programming languages? Python, Java, or JavaScript are all popular ones. What's the most popular programming language right now? Python is very popular, but others like Java and C++ are also used. Do you think programming is easy to learn? Yes, it is. Many people start with programming because it's fun and easy to learn. Do you have any tips for learning programming? Start with simple tasks and practice, and don't be afraid to ask for help if you need it. Good luck with your haiku. Have fun with it!

### 2. Write a haiku about programming in 5 lines

Write a haiku about programming in 5 lines.

I'm going to write a haiku about programming in 5 lines. What's programming? 

## 💐 You're done!

This notebook provided a step-by-step guide to fine-tuning the `HuggingFaceTB/SmolLM2-135M` model using the `SFTTrainer`. By following these steps, you can adapt the model to perform specific tasks more effectively. If you want to carry on working on this course, here are steps you could try out:

- Try this notebook on a harder difficulty
- Review a colleagues PR
- Improve the course material via an Issue or PR.

see [output of first haiku](#haiku-output-1)

In [ ]:
!git diff

usage: git diff --no-index [<options>] <path> <path>

Diff output format options
    -p, --patch           generate patch
    -s, --no-patch        suppress diff output
    -u                    generate patch
    -U, --unified[=<n>]   generate diffs with <n> lines context
    -W, --function-context
                          generate diffs with <n> lines context
    --raw                 generate the diff in raw format
    --patch-with-raw      synonym for '-p --raw'
    --patch-with-stat     synonym for '-p --stat'
    --numstat             machine friendly --stat
    --shortstat           output only the last line of --stat
    -X, --dirstat[=<param1,param2>...]
                          output the distribution of relative amount of changes for each sub-directory
    --cumulative          synonym for --dirstat=cumulative
    --dirstat-by-file[=<param1,param2>...]
                          synonym for --dirstat=files,param1,param2...
    --check               warn if changes introduce

In [29]:
haiku_ds = load_dataset("statworx/haiku")

In [31]:
print(haiku_ds)
print(haiku_ds['train'][3])

DatasetDict({
    train: Dataset({
        features: ['source', 'text', 'text_phonemes', 'keywords', 'keyword_phonemes', 'gruen_score', 'text_punc'],
        num_rows: 49024
    })
})
{'source': 'bfbarry', 'text': 'You were broken glass. / But I touched you even though. / I knew it would hurt.', 'text_phonemes': 'yuw wer brow|kaxn glaes / baht ay tahcht yuw iy|vaxn dhow / ay nuw iht wuhd hhert', 'keywords': 'broken glass', 'keyword_phonemes': 'brow|kaxn glaes', 'gruen_score': 0.703445971, 'text_punc': None}


In [38]:
def process_dataset(sample):
    messages = [
        {"role": "user", "content": f"Write a haiku about {sample['keywords']}"},
        {"role": "assistant", "content": sample["text"].replace("/", "\n")},
    ]
    # apply the chat template to the samples using the tokenizer's method
    sample['messages'] = messages
    return sample


haiku_ds = haiku_ds.map(process_dataset)


Map:   0%|          | 0/49024 [00:00<?, ? examples/s]

In [41]:
print(haiku_ds)
print(haiku_ds['train'][0])

DatasetDict({
    train: Dataset({
        features: ['source', 'text', 'text_phonemes', 'keywords', 'keyword_phonemes', 'gruen_score', 'text_punc', 'messages'],
        num_rows: 49024
    })
})
{'source': 'bfbarry', 'text': "Delicate savage. / You'll never hold the cinder. / But still you will burn.", 'text_phonemes': 'deh|lax|kaxt sae|vaxjh / yuwl neh|ver hhowld dhax sihn|der / baht stihl yuw wihl bern', 'keywords': 'cinder', 'keyword_phonemes': 'sihn|der', 'gruen_score': 0.639071301, 'text_punc': None, 'messages': [{'content': 'Write a haiku about cinder', 'role': 'user'}, {'content': "Delicate savage. \n You'll never hold the cinder. \n But still you will burn.", 'role': 'assistant'}]}


In [43]:
haiku_ds = haiku_ds['train'].train_test_split(test_size=0.2)

In [45]:
print(haiku_ds['train'][0])
print(haiku_ds['test'][0])

{'source': 'twaiku', 'text': 'Ray Liotta quit. / Smoking good on him, where do? / I insert the jokes.', 'text_phonemes': 'rey liy|ow|tax kwiht / smow|kaxng guhd aan hhihm wehr duw / ay axn|sert dhax jhowks', 'keywords': 'the', 'keyword_phonemes': 'dhax', 'gruen_score': 0.698370323, 'text_punc': None, 'messages': [{'content': 'Write a haiku about the', 'role': 'user'}, {'content': 'Ray Liotta quit. \n Smoking good on him, where do? \n I insert the jokes.', 'role': 'assistant'}]}
{'source': 'twaiku', 'text': 'Even if they were. / Good, no one was coming out. / To see McNeese state.', 'text_phonemes': 'iy|vaxn axf dhey wer / guhd now wahn waaz kah|maxng awt / tax siy maxk|niys steyt', 'keywords': 'they were', 'keyword_phonemes': 'dhey wer', 'gruen_score': 0.666841167, 'text_punc': None, 'messages': [{'content': 'Write a haiku about they were', 'role': 'user'}, {'content': 'Even if they were. \n Good, no one was coming out. \n To see McNeese state.', 'role': 'assistant'}]}


In [46]:
from trl import SFTConfig, SFTTrainer

# Set our name for the finetune to be saved &/ uploaded to
finetune_name = "SmolLM2-FT-MyDataset-Haiku"

# Configure the SFTTrainer
# report_to="none" to disable wandb
sft_config = SFTConfig(
    output_dir="./sft_output-haiku",
    max_steps=1000,  # Adjust based on dataset size and desired training duration
    per_device_train_batch_size=4,  # Set according to your GPU memory capacity
    learning_rate=5e-5,  # Common starting point for fine-tuning
    logging_steps=10,  # Frequency of logging training metrics
    save_steps=100,  # Frequency of saving model checkpoints
    evaluation_strategy="steps",  # Evaluate the model at regular intervals
    eval_steps=50,  # Frequency of evaluation
    use_mps_device=(
        True if device == "mps" else False
    ),  # Use MPS for mixed precision training
    hub_model_id=finetune_name,  # Set a unique name for your model
    report_to="none"
)

# Initialize the SFTTrainer
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=haiku_ds["train"],
    tokenizer=tokenizer,
    eval_dataset=haiku_ds["test"]
)

# TODO: 🦁 🐕 align the SFTTrainer params with your chosen dataset. For example, if you are using the `bigcode/the-stack-smol` dataset, you will need to choose the `content` column`

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-46-aa6ae2c18668>:25: FutureWarning: `tokenizer` is deprecated and removed starting from version 0.16.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  trainer = SFTTrainer(


Map:   0%|          | 0/39219 [00:00<?, ? examples/s]

Map:   0%|          | 0/9805 [00:00<?, ? examples/s]

In [47]:
# Train the model
trainer.train()

# Save the model
trainer.save_model(f"./{finetune_name}")

Step,Training Loss,Validation Loss
50,2.214700,2.170475
100,2.049700,2.129663
150,2.034800,2.101195
200,2.012000,2.091916
250,2.008300,2.077918
300,1.997300,2.077035
350,1.981000,2.056500
400,1.885200,2.058964
450,2.016300,2.046031
500,2.088100,2.036363


In [49]:
# Test the fine-tuned model on the same prompt

# Let's test the base model before training
prompt = "Write a haiku about programming"

# Format with template
messages = [{"role": "user", "content": prompt}]
formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False)

# Generate response
inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)

# TODO: use the fine-tuned to model generate a response, just like with the base example.

In [51]:
outputs = model.generate(**inputs, max_new_tokens=100)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

user
Write a haiku about programming
assistant
I'm not programming. 
 But I'm trying to learn. 
 How to program. 
 I'm not programming. 
 I'm trying to learn. 
 How to program. 
 I'm not programming. 
 I'm trying to learn. 
 How to program. 
 I'm not programming. 
 I'm trying to learn. 
 How to program. 
 I'm not programming. 
 I'm trying to learn. 
 How to program. 
 I'm not programming.


In [52]:
prompt = "What is a haiku?"

# Format with template
messages = [{"role": "user", "content": prompt}]
formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False)

# Generate response
inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)

outputs = model.generate(**inputs, max_new_tokens=100)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

user
What is a haiku?
What is a haiku? A.
Haiku is a word. 
 That means a lot of things. 
 I don't know what. 
 I'm doing with. 
 It, I don't know. 
 What I'm doing with. 
 It, I don't know. 
 What I'm doing with. 
 It, I don't know. 
 What I'm doing with. 
 It, I don't know. 
 What I'm doing with. 
 It, I


In [54]:
prompt="What is a Haiku?"
inputs = tokenizer.encode(prompt, return_tensors="pt").to(device)
outputs = model.generate(inputs, max_new_tokens=100)
print(tokenizer.decode(outputs[0]))

What is a Haiku?

A Haiku is a short poem that is about a single word. It is a short poem that is about a single word. It is a short poem that is about a single word. It is a short poem that is about a single word. It is a short poem that is about a single word. It is a short poem that is about a single word. It is a short poem that is about a single word. It is a short poem that is about a single word.
